### Project - 

In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import json
import logging
import copy
from copy import deepcopy
import random
import functools
from typing import Optional, Dict, Sequence, List
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

import transformers
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    PreTrainedTokenizerFast,
    GPT2Config,
    GPT2Model,
    pipeline
)

from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

import guide

In [ ]:
try:
    root_path = os.path.dirname(os.path.abspath(__file__))
except:
    root_path = os.getcwd()
    
cfg = {
    "model_name": "skt/kogpt2-base-v2",
    # "device": "cuda" if torch.cuda.is_available() else "cpu",
    "device": torch.device("xpu" if torch.xpu.is_available() else "cuda" if torch.cuda.is_available() else "cpu"),
    "root_path": root_path,

    "sft_output_dir": root_path + "/test",
    "sft_saved_dir": root_path + "/models/output_1_SFT",
    "sft_num_train_epochs": 1,
    "sft_per_device_train_batch_size": 4,
    "sft_per_device_eval_batch_size": 4,
    "sft_warmup_steps": 5,
    "sft_prediction_loss_only": True,
    "sft_fp16": False # XPU sometimes has issues with fp16 in older versions or specific setups, let's try False or BF16
}

# tokenizer, model 준비
model = AutoModelForCausalLM.from_pretrained(cfg["model_name"]).to(cfg["device"])
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    cfg["model_name"],
    bos_token='</s>', eos_token='</s>', unk_token='<unk>', pad_token='<pad>', mask_token='<mask>',
    padding_side="right",
    model_max_length=512,
)

guide.step_02_show_info(cfg)
# show_base_model_and_dataset(cfg, model, tokenizer)
# show_sft_and_rm_dataset(cfg)
# guide.run_sft(cfg, model, tokenizer)
# guide.run_reward_model(cfg)
# guide.run_ppo(cfg, model, tokenizer)

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 66947.11it/s]
The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



-------------------- step_02_show_info() --------------------
Torch version: 2.10.0+cu128
Device: cpu
transformers version: 5.3.0
Using CPU


In [ ]:
# !pip install lm-eval

!lm-eval --model hf \
    --model_args pretrained=models/output_1_SFT \
    --tasks mmlu,gsm8k

    # --device cuda:0 \
    # --batch_size 8
    
    # !lm-eval --model hf \
    # --model_args pretrained=models/output_1_SFT\
    # --tasks mmlu,gsm8k
    
    # --device cuda:0 \
    # --batch_size 8

2026-03-14:06:35:48 INFO     [_cli.run:376] Selected Tasks: ['mmlu', 'gsm8k']
2026-03-14:06:35:49 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-14:06:35:49 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'models/output_1_SFT'}
2026-03-14:06:35:50 INFO     [models.huggingface:169] Device not specified
2026-03-14:06:35:50 INFO     [models.huggingface:170] Cuda Available? False
2026-03-14:06:35:50 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cpu'}
Loading weights: 100%|█████████████████████| 149/149 [00:00<00:00, 16338.17it/s]
The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silen